# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

In [3]:
import os
from uuid import uuid4


os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"Retrieval Evaluation - {uuid4().hex[0:8]}"

os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [5]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

In [6]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

synthetic_usecase_data = docs

Let's look at an example document to see if everything worked as expected!

In [7]:
synthetic_usecase_data[0]

Document(metadata={'producer': 'macOS Version 15.4.1 (Build 24E263) Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-12T20:05:32+00:00', 'source': 'data/howpeopleuseai.pdf', 'file_path': 'data/howpeopleuseai.pdf', 'total_pages': 64, 'format': 'PDF 1.6', 'title': 'How People Use ChatGPT', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-15T10:32:36-04:00', 'trapped': '', 'modDate': "D:20250915103236-04'00'", 'creationDate': 'D:20250912200532Z', 'page': 0}, page_content='NBER WORKING PAPER SERIES\nHOW PEOPLE USE CHATGPT\nAaron Chatterji\nThomas Cunningham\nDavid J. Deming\nZoe Hitzig\nChristopher Ong\nCarl Yan Shan\nKevin Wadman\nWorking Paper 34255\nhttp://www.nber.org/papers/w34255\nNATIONAL BUREAU OF ECONOMIC RESEARCH\n1050 Massachusetts Avenue\nCambridge, MA 02138\nSeptember 2025\nWe acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan \nBeiermeister, Rachel Brown, Cassandra Duchan Solis, Jason Kwon, 

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [8]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [9]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [11]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [12]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [61]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project or activity domain for ChatGPT usage, based on the provided data, is related to work activities that involve obtaining, documenting, interpreting information, and making decisions or solving problems. Specifically, the most frequently associated work activities include activities like seeking practical guidance, writing, and technical help, which are central to various professional and knowledge-based projects.\n\nIn summary, the most common project domain appears to be related to professional or work-oriented activities centered around information processing, decision support, and communication tasks across different occupations.'

In [11]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. One example is the project titled "WealthifyAI," which involves a federated learning toolkit aimed at improving privacy in healthcare applications. Additionally, the project "Pathfinder 24" focuses on an AI-powered platform for optimizing logistics routes for sustainability, which has potential security considerations in its scalable and well-structured design.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges generally had positive comments about the fintech-related projects, highlighting their technical ambition, robustness, and real-world impact. For example, the project "GreenPulse" in the FinTech secondary domain was described as "Technically ambitious and well-executed," receiving a judge score of 8.9. Similarly, "PixelSense" was noted for having a "Comprehensive and technically mature approach," with a judge score of 8.4. Additionally, "DataWeave" was praised for "Excellent code quality and use of open-source libraries," earning a high judge score of 9.8. Overall, judges appreciated the innovation, quality, and potential real-world benefits of the fintech projects.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [72]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [73]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [74]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain is work-related messages focusing on core job tasks such as decision-making, problem-solving, documenting information, writing, and technical help.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. The project titled "SecureNest 49" involves a document summarization and retrieval system for enterprise knowledge bases, which falls under the secondary domain of Legal / Compliance.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had the following comments about the fintech projects:\n\n- For the project "SynthMind" in the Finance / FinTech domain, the judges found it to be "conceptually strong but results need more benchmarking," indicating approval of the idea but suggesting the need for more performance validation.\n  \nOverall, the feedback suggests that the fintech projects were viewed positively in terms of their conceptual strength, although some comments pointed to areas for improvement such as benchmarking and evaluation.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

BM25 retriever is used for exact word matching search. A great fit would be for example scientific or technical papers. Query examples: "What is Bernoulli's law?" or "What is the specific error code 404 for HTTP?".


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [75]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [76]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [77]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain, based on the provided context, appears to be related to work activities involving obtaining, documenting, interpreting information, making decisions, giving advice, solving problems, and thinking creatively. Specifically, activities such as "documenting/recording information" and "making decisions and solving problems" are highlighted as prevalent across occupations.'

In [27]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicit use cases related to security. The projects mentioned focus on federated learning to improve privacy in healthcare applications, but there is no specific mention of security use cases.'

In [28]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects were positive. For example, in the case of the project "Pathfinder 27" in the Finance / FinTech domain, the judges praised the project for its "excellent code quality and use of open-source libraries."'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [78]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [79]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [80]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project or activity domain related to ChatGPT usage, across various occupation groups and conversation topics, appears to be focused on information-related activities such as making decisions, solving problems, documenting and recording information, and interpreting the meaning of information for others. These activities are consistently among the top work activities associated with ChatGPT across diverse occupations.\n\nSpecifically, the most prevalent project domains include:\n- obtaining and documenting information\n- making decisions and solving problems\n- interpreting and recording information\n- providing consultation and advice\n- thinking creatively\n\nOverall, the dominant project domain is centered on information processing, decision support, and knowledge work, reflecting that the primary use of ChatGPT across professions involves inquiry, documentation, analysis, and advice, rather than specialized tasks like coding or social-emotional support.'

In [33]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security in the provided context. For example, there is a project called "SecureNest" that involves a document summarization and retrieval system for enterprise knowledge bases, which pertains to legal and compliance aspects of security.'

In [34]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges generally had positive feedback about the fintech projects. They described some projects as having strong conceptual frameworks, robust experimentation, and impressive real-world impact. For example, one project was praised as "Solid work with impressive real-world impact," and another was noted as "Technically ambitious and well-executed." Additionally, some projects received high judge scores, such as 9.6 and 9.2, indicating strong approval from the judges. Overall, the judges acknowledged the projects\' technical maturity, promising ideas, and potential for scalable and impactful solutions.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

If one query fails, another one may succeed. For example a query for "fast CPU" might be reformulated into "high-speed processor" or "What is RAG?" into "Define Retrieval-Augmented Generation".

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [81]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [82]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [83]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [84]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [85]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [86]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain, based on the usage patterns of ChatGPT described in the context, is "Writing." It accounts for a significant portion of both work-related and general conversations, including activities like modifying, editing, or generating text, and is the leading topic overall.'

In [41]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases about security mentioned. The projects mainly focus on federated learning and privacy in healthcare applications, but security is not explicitly highlighted as a key use case.'

In [42]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges made positive remarks about the fintech-related projects. For example, they commented that one project was a "clever solution with measurable environmental benefit," and another was described as having a "robust experimental validation." Overall, the judge comments highlighted the projects\' innovation, real-world impact, and thoroughness.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [87]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [88]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [89]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'seeking_information'

In [46]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was at least one use case about security. Specifically, the project "SecureNest 49" is described as a "document summarization and retrieval system for enterprise knowledge bases," which relates to security in the context of information protection and data management.'

In [47]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had a generally positive view of the fintech projects. For example, the project "PulseAI," which is an adaptive fine-tuning pipeline for multilingual reasoning models in the fintech domain, was described as "technically ambitious and well-executed" and received a high judge score of 8.0. Additionally, "DocuCheck," an AI-powered platform optimizing logistics routes for sustainability in fintech, was considered "conceptually strong" with a very high judge score of 9.6. Overall, judges appreciated the technical strength, innovative ideas, and real-world impact of the fintech projects, though some noted areas for further benchmarking and stronger evaluation metrics.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [91]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [92]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [93]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [94]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [95]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [96]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain, based on the provided context, appears to be related to "Writing." ChatGPT is frequently used for writing tasks such as editing, critiquing, translating, summarizing, and generating content. Writing consistently accounts for a significant portion of user interactions, especially in work-related messages, where it can comprise up to 40% of all work-related exchanges and is highlighted as a dominant activity across various analyses. Additionally, activities like Practical Guidance and Seeking Information are common, but "Writing" specifically stands out as the primary focus in many contexts, especially for work tasks.\n\nTherefore, the most common project domain is **Writing**.'

In [55]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the project "BioForge" is described as a medical imaging solution improving early diagnosis through vision transformers, and it is categorized under the Security domain. Additionally, "InsightAI" is a low-latency inference system for multimodal agents in autonomous systems, also listed under the Security domain.'

In [56]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had various comments about the fintech projects. For example:\n\n- "TrendLens 19" was described as "Technically ambitious and well-executed."\n- "WealthifyAI 16" was called a "Comprehensive and technically mature approach."\n- "AutoMate 5" was noted as "A forward-looking idea with solid supporting data."\n- "InsightAI 1" received praise for being "Technically ambitious and well-executed."\n\nOverall, the judges appreciated the technical ambition, execution quality, and potential of the fintech projects.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

If sentences are short and highly repetitive that means they would have similar embeddings. Semantic chuncking algorithm probably wont find a "breakpoint" and will merge everything into a large single chunk.

The solution is to use another chuncking strategy or change breakpoint threshold value.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

#### create llm and embedding models

In [97]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

#### load docs

In [98]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

#### create dataset

In [99]:
from ragas.testset import TestsetGenerator

# docs = synthetic_usecase_data

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=5)

dataset.to_pandas()

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

Property 'summary' already exists in node '420065'. Skipping!
Property 'summary' already exists in node '0dc0c0'. Skipping!
Property 'summary' already exists in node '4de578'. Skipping!
Property 'summary' already exists in node 'eee7bb'. Skipping!
Property 'summary' already exists in node '039f8e'. Skipping!
Property 'summary' already exists in node '0ca364'. Skipping!
Property 'summary' already exists in node 'c9dad7'. Skipping!
Property 'summary' already exists in node '5106b5'. Skipping!
Property 'summary' already exists in node '5c7925'. Skipping!
Property 'summary' already exists in node '64a880'. Skipping!
Property 'summary' already exists in node '391757'. Skipping!
Property 'summary' already exists in node '5a60f0'. Skipping!
Property 'summary' already exists in node 'd65ee7'. Skipping!
Property 'summary' already exists in node 'ae4010'. Skipping!
Property 'summary' already exists in node '651b35'. Skipping!
Property 'summary' already exists in node 'bccbe6'. Skipping!
Property

...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,what Acemoglu say about AI growth?,[Introduction ChatGPT launched in November 202...,The sudden growth in LLM abilities and adoptio...,single_hop_specifc_query_synthesizer
1,How many 700 million users were using ChatGPT ...,[Conclusion This paper studies the rapid growt...,"By July 2025, ChatGPT had been used weekly by ...",single_hop_specifc_query_synthesizer
2,how chatgpt adoption and usage statistics show...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,ChatGPT launched in November 2022 and by July ...,multi_hop_abstract_query_synthesizer
3,What do the ChatGPT adoption and usage statist...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had experienced unpreced...",multi_hop_abstract_query_synthesizer
4,Considering the rapid global diffusion of Chat...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had rapidly diffused glo...",multi_hop_specific_query_synthesizer
5,How has ChatGPT's rapid adoption by 700 millio...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had rapidly grown to 700...",multi_hop_specific_query_synthesizer


#### evaluate

In [100]:
import copy
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    ResponseRelevancy,
    Faithfulness,
    AnswerRelevancy
)
from ragas import evaluate, RunConfig

def eval_chain(chain, dataset):
    evaluation_dataset = copy.deepcopy(dataset)

    for test_row in evaluation_dataset:

        result = chain.invoke({"question" : test_row.eval_sample.user_input})

        test_row.eval_sample.response = result["response"].content
        test_row.eval_sample.retrieved_contexts = [context.page_content for context in result["context"]]

    evaluation_dataset = EvaluationDataset.from_pandas(evaluation_dataset.to_pandas())
    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

    evaluation_result = evaluate(
        dataset=evaluation_dataset,
        metrics=[
                ContextPrecision(),
                ContextRecall(),
                ResponseRelevancy(),
                Faithfulness(),
                AnswerRelevancy()
            ],
        llm=evaluator_llm,
        run_config=RunConfig(timeout=720)
    )

    return evaluation_result

evaluation_result = eval_chain(naive_retrieval_chain, dataset)


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

In [146]:
evaluation_result

{'context_precision': 0.9623, 'context_recall': 1.0000, 'answer_relevancy': 0.9337, 'faithfulness': 0.9339}

In [155]:

def eval_to_scores(evaluation_result):
    
    mean_scores = {}
    keys = evaluation_result.scores[0].keys()
    for key in keys:
        values = [float(score[key]) if hasattr(score[key], "item") else score[key] for score in evaluation_result.scores if key in score]
        mean_scores[key] = sum(values) / len(values) if values else None

    return mean_scores

mean_scores = eval_to_scores(evaluation_result)

mean_scores

{'context_precision': 0.9622767856877182,
 'context_recall': 1.0,
 'answer_relevancy': 0.933683064388582,
 'faithfulness': 0.9339293886360961}

In [158]:
from langsmith import Client
import time

ls_client = Client()

def get_last_n_runs_metrics(n=5):

    # delay to sync langsmith client
    time.sleep(5)

    runs = ls_client.list_runs(
        project_name=os.environ["LANGCHAIN_PROJECT"],
        filter='and(eq(is_root, true), neq(name, "ragas evaluation"))',
        limit=n
    )

    acc_latency = 0
    acc_cost = 0
    acc_tokens = 0

    iters = 0;

    for run in runs:
        acc_latency += run.latency
        acc_cost += run.total_cost
        acc_tokens += run.total_tokens

        print("total_cost: ", run.total_cost)
        print("latency: ", run.latency)
        print("total_tokens: ", run.total_tokens)
        print("--------------------------------")

        iters += 1

    metrics = {
            "latency": acc_latency / iters, 
            "cost": acc_cost / iters, 
            "tokens": acc_tokens / iters
        }

    return metrics;


In [139]:
metrics = get_last_n_runs_metrics(5)

metrics

total_cost:  0.0009905
latency:  3.442072
total_tokens:  8921
--------------------------------
total_cost:  0.001074
latency:  5.714107
total_tokens:  9237
--------------------------------
total_cost:  0.0011267
latency:  4.831075
total_tokens:  9440
--------------------------------
total_cost:  0.0008657
latency:  -1.037526
total_tokens:  8498
--------------------------------
total_cost:  0.0010979
latency:  6.672857
total_tokens:  8999
--------------------------------


{'latency': 3.9245170000000003,
 'cost': Decimal('0.00103096'),
 'tokens': 9019.0}

In [154]:

total_metrics = {**mean_scores, **metrics}

total_metrics

{'context_precision': 0.9622767856877182,
 'context_recall': 1.0,
 'answer_relevancy': 0.933683064388582,
 'faithfulness': 0.9339293886360961,
 'latency': 3.9245170000000003,
 'cost': Decimal('0.00103096'),
 'tokens': 9019.0}

In [156]:
def evaluate_retrieval_chain(chain, dataset, n_runs=5):
    """
    Evaluate a retrieval chain and return comprehensive metrics.
    
    Args:
        chain: The LangChain retrieval chain to evaluate
        dataset: The test dataset generated by Ragas
        n_runs: Number of recent runs to get metrics from LangSmith (default: 5)
    
    Returns:
        dict: Combined metrics including:
            - Ragas evaluation metrics (context_precision, context_recall, 
              response_relevancy, faithfulness, answer_relevancy)
            - LangSmith metrics (latency, cost, tokens)
    """
    # Step 1: Evaluate chain with Ragas
    evaluation_result = eval_chain(chain, dataset)
    
    # Step 2: Convert evaluation results to mean scores
    mean_scores = eval_to_scores(evaluation_result)
    
    # Step 3: Get LangSmith metrics
    langsmith_metrics = get_last_n_runs_metrics(n=n_runs)
    
    # Step 4: Combine all metrics
    total_metrics = {**mean_scores, **langsmith_metrics}
    
    return total_metrics


In [159]:
# Evaluate a single chain
total_metrics = evaluate_retrieval_chain(naive_retrieval_chain, dataset, n_runs=5)
print(total_metrics)

# Or evaluate multiple chains
chains_to_evaluate = {
    "naive": naive_retrieval_chain,
    "bm25": bm25_retrieval_chain,
    "compression": contextual_compression_retrieval_chain,
    "multi_query": multi_query_retrieval_chain,
    "parent_document": parent_document_retrieval_chain,
    "ensemble": ensemble_retrieval_chain,
    "semantic_chunking": semantic_retrieval_chain
}

results = {}
for name, chain in chains_to_evaluate.items():
    print(f"Evaluating {name}...")
    results[name] = evaluate_retrieval_chain(chain, dataset, n_runs=5)

Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

total_cost:  0.0009609
latency:  2.784498
total_tokens:  8847
--------------------------------
total_cost:  0.0011008
latency:  5.189504
total_tokens:  9304
--------------------------------
total_cost:  0.0011631
latency:  5.749829
total_tokens:  9531
--------------------------------
total_cost:  0.0010611
latency:  5.340511
total_tokens:  8907
--------------------------------
total_cost:  0.0008713
latency:  1.32964
total_tokens:  8512
--------------------------------
{'context_precision': 0.9731481481211085, 'context_recall': 1.0, 'answer_relevancy': 0.7899001120409492, 'faithfulness': 0.8301102839812518, 'latency': 4.0787964, 'cost': Decimal('0.00103144'), 'tokens': 9020.2}
Evaluating naive...


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

total_cost:  0.0003705
latency:  2.424803
total_tokens:  8883
--------------------------------
total_cost:  0.0010792
latency:  1.747477
total_tokens:  9250
--------------------------------
total_cost:  0.0005163
latency:  3.712852
total_tokens:  9426
--------------------------------
total_cost:  0.0004127
latency:  3.513045
total_tokens:  8798
--------------------------------
total_cost:  0.0008713
latency:  1.251219
total_tokens:  8512
--------------------------------
Evaluating bm25...


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

...

total_cost:  0.0002644
latency:  1.661675
total_tokens:  1945
--------------------------------
total_cost:  0.000374
latency:  -0.981961
total_tokens:  2543
--------------------------------
total_cost:  0.000395
latency:  1.789786
total_tokens:  3095
--------------------------------
total_cost:  0.000415
latency:  2.696471
total_tokens:  2920
--------------------------------
total_cost:  0.0003626
latency:  1.072466
total_tokens:  3494
--------------------------------
Evaluating ensemble...


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

total_cost:  0.0013899
latency:  6.048044
total_tokens:  12693
--------------------------------
total_cost:  0.001355
latency:  6.87654
total_tokens:  12920
--------------------------------
total_cost:  0.0011985
latency:  4.93254
total_tokens:  10068
--------------------------------
total_cost:  0.001442
latency:  8.068688
total_tokens:  12269
--------------------------------
total_cost:  0.000839
latency:  5.732941
total_tokens:  10079
--------------------------------
Evaluating semantic_chunking...


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

total_cost:  0.0007785
latency:  2.313039
total_tokens:  6987
--------------------------------
total_cost:  0.0008264
latency:  3.333545
total_tokens:  6983
--------------------------------
total_cost:  0.0009617
latency:  6.299121
total_tokens:  7466
--------------------------------
total_cost:  0.0008701
latency:  4.620804
total_tokens:  6919
--------------------------------
total_cost:  0.0005783
latency:  0.834234
total_tokens:  5657
--------------------------------


In [160]:
results

{'naive': {'context_precision': 0.9669238682854164,
  'context_recall': 1.0,
  'answer_relevancy': 0.7890393952840719,
  'faithfulness': 0.8238502238502239,
  'latency': 2.5298792000000003,
  'cost': Decimal('0.0006500'),
  'tokens': 8973.8},
 'bm25': {'context_precision': 0.8194444444226852,
  'context_recall': 0.8055555555555555,
  'answer_relevancy': 0.9482655862890219,
  'faithfulness': 0.8083134920634921,
  'latency': 2.5237726,
  'cost': Decimal('0.00051716'),
  'tokens': 4030.4},
 'compression': {'context_precision': 0.9999999999555556,
  'context_recall': 1.0,
  'answer_relevancy': 0.7822656460754741,
  'faithfulness': 0.9124384236453201,
  'latency': 10.715238,
  'cost': Decimal('0.00043632'),
  'tokens': 3357.6},
 'multi_query': {'context_precision': 0.8726424424209448,
  'context_recall': 0.9722222222222222,
  'answer_relevancy': 0.7946677868100003,
  'faithfulness': 0.9756003927515556,
  'latency': 4.4128118,
  'cost': Decimal('0.0012767'),
  'tokens': 11267.0},
 'parent_do

Try to rerun evaluation with different queries

#### Summary

See eval_results folder

- BM25 gave the most relevant answers (answer relevancy ≈ 0.95) and was fast and cheap, making it ideal for keyword-heavy or FAQ-style data.

- Parent-document had the best balance overall — very fast (1.25 s), low cost, and strong faithfulness (0.90). It suits structured or long documents.

- Compression achieved near-perfect recall and high faithfulness, but was slow, useful for high-accuracy, low-volume tasks.

- Multi-query produced the most faithful answers (0.98) by retrieving from multiple angles, but was expensive and slower — best for ambiguous or complex queries.

- Naive and Ensemble covered everything (recall = 1.0) but added extra cost and latency without improving answers much.

Overall, for this dataset, Parent-document is the best all-around retriever because it provides high precision and faithfulness at the lowest latency and cost.
Use BM25 for fast, lexical lookups, and Compression or Multi-query when accuracy and faithfulness are more important than speed.
    

- Without semantic chunking, naive retrieval gives higher recall and precision, retrieving nearly all relevant info efficiently.

- With semantic chunking, the model retrieves smaller, semantically cleaner segments, which slightly improves faithfulness (fewer hallucinations) but hurts precision and recall, and increases latency.